In [125]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget
import matplotlib.pyplot as plt


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.min_halo = 7.
        self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)

        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self): ##converting this to ln????
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
        
    
    def get_slope(self, Mhalo): #returns dlogMstar/dlogMhalo slope is same in log10 and ln space

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sig_lnMstar, bin_num, z): 
        lnten = np.log(10)
        logMh = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sig_lnMstar) / self.get_slope(np.asarray(logMh)) ##is this the right sigma?
        mins = logMh * lnten - plus_mins
        maxs = logMh * lnten + plus_mins
        maxMh, minMh = self.min_halo * lnten, self.max_halo * lnten
        mins[mins < maxMh] = maxMh
        maxs[maxs > minMh] = minMh
        lnMh = create_ranges_numexpr(mins, maxs, bin_num)
        dNdlnMhalo = mf.massFunction(np.e**lnMh, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        logMstar = np.apply_along_axis(self.get_Mstar, 1, lnMh/lnten)
        vals = np.zeros((bin_num,bin_num+1))
        vals[:,-1] = StellBins * lnten
        vals[:,:-1] = logMstar * lnten
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, vals, sig_lnMstar, 1)
        dNdlnMstar = np.sum(Mstar_prob * dNdlnMhalo, axis = 1) * (lnMh[:,1] - lnMh[:,0])

        return dNdlnMstar
    
    
    
    def get_dNdlnMstar(self, sig_lnMstar):
        
        if sig_lnMstar == 0.:
            self.dNdlnMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        else:
            self.dNdlnMstar = self.convolve_smhm(self.StellBins, sig_lnMstar, self.bin_num, self.z)


            
    def get_dNdlnMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdlnMbh = self.dNdlnMstar / self.m
        
    def lnetas(self, lnMbh):
    
        lneta = np.asarray(self.LumBins) * np.log(10) - np.log(3.3e4) - lnMbh

        return lneta
        
    
    def get_mu_lneta(self, vals, a, lnxsigs, files = files):

        Mbh = np.e**vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            lnxsig = lnxsigs[0]
        else:
            lnxsig = lnxsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3e38 * Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99e10)**2) #kg/s
        sbhr = slope * (ssfr / 3.154e7) #1/s
        Mdotbh = sbhr * (Mbh * 2e33)
        eta = Mdotbh/Mdotedd
        
        
        mu_lnX = -0.5 * lnxsig**2
        mu_lnMdotbh = mu_lnX + np.log(Mdotbh / 2e33)
        mu_lneta = mu_lnMdotbh - np.log(Mdotedd / 2e33)
        
        lnetasig = lnxsig
        
        return mu_lneta, lnetasig
    
    def gauss(self, x, *var):
  
        mu, sig, A = var
        y = ( A/np.sqrt(2.0 * np.pi * sig**2.0) ) * np.exp( -(x - mu)**2.0 / (2.0 * sig**2) )

        return y
    
    
    def prob_lneta(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2], vals[-1], 1)

        return probdens
    
    def get_dNdlnL(self, lnxsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0] * np.log(10)
        b[self.growth] = self.int_list[1] * np.log(10)
        b[self.late] = self.int_list[2] * np.log(10)

        lnBHBins = self.StellBins * np.log(10) * self.m + b
        lneta_lists = np.apply_along_axis(self.lnetas, 1, np.reshape(lnBHBins,(self.bin_num,1)))
        

        vals = np.zeros((self.bin_num,3))
        vals[:,0] = lnBHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mu_sig_lnetas = np.apply_along_axis(self.get_mu_lneta, 1, vals, self.a, lnxsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = lneta_lists
        vals[:,-2] = mu_sig_lnetas[:,0]
        vals[:,-1] = mu_sig_lnetas[:,1]
            
        ##why are we multiplying by slope???
        self.dNdlnL = (1-obscured) * (np.sum(np.apply_along_axis(self.prob_lneta, 1, vals) * np.reshape(self.dNdlnMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0))
        ind_dNdlnL_off = (1-obscured) * np.apply_along_axis(self.prob_lneta, 1, vals) * np.reshape(self.dNdlnMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdlnL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdlnL_off:
            self.ind_dNdlnL[c,:] = l
            c += 1

bins = 500
fig = plt.figure(figsize=(5,5))
z=3
obscured = .5
sig_lnX = [2.5,2.3]
dM = .2
mMid = 10.3
sig_lnMstar = 0.3*np.log(10)


def get_QLF():
    qlf = QLF(z, bins)
    qlf.get_dNdlnMstar(sig_lnMstar)
    qlf.get_SMBM(dM, mMid)
    qlf.get_dNdlnMbh()
    qlf.get_dNdlnL(sig_lnX, obscured)

for i in range(10):
    
    %time get_QLF()
col = np.zeros(qlf.bin_num)
col[qlf.early] = 0
col[qlf.growth] = 1
col[qlf.late] = 2


plt.plot(qlf.LumBins, np.log10(qlf.dNdlnL * np.log(10)),c='k',label='z = '+str(z))
plt.axis([5.8,16,-10,0])

# qlf.get_dNdlnMstar(0)
# qlf.get_SMBM(dM, mMid)
# qlf.get_dNdlnMbh()
# qlf.get_dNdlnL(sig_lnX, obscured)

# plt.plot(qlf.LumBins, np.log10(qlf.dNdlnL * np.log(10)),c='orange',label='z = '+str(z))

x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='gold',label='observed')
count1 = 0
count2 = 0
count3 = 0
for l, c in zip(qlf.ind_dNdlnL, col):
    c_list = ['r','g','b']
    if c == 0 and count1 == 0:
        plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='pre-disk: early')
        count1 += 1
    elif c == 1 and count2 == 0:
        plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='post-disk: growth')
        count2 += 1
    elif c == 2 and count3 == 0:
        plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='post-disk: late')
        count3 += 1
    else:
        plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)])
plt.xlabel(r'$\log_{10} [L_{bol}/L_{\odot}]$')
plt.ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$')
plt.legend()



FigureCanvasNbAgg()

CPU times: user 183 ms, sys: 13.6 ms, total: 197 ms
Wall time: 234 ms
CPU times: user 155 ms, sys: 6.81 ms, total: 162 ms
Wall time: 171 ms
CPU times: user 165 ms, sys: 8.3 ms, total: 173 ms
Wall time: 186 ms
CPU times: user 157 ms, sys: 6.57 ms, total: 163 ms
Wall time: 166 ms
CPU times: user 153 ms, sys: 6.15 ms, total: 159 ms
Wall time: 165 ms
CPU times: user 182 ms, sys: 8.48 ms, total: 191 ms
Wall time: 197 ms
CPU times: user 194 ms, sys: 8.34 ms, total: 202 ms
Wall time: 212 ms
CPU times: user 185 ms, sys: 7.68 ms, total: 192 ms
Wall time: 195 ms
CPU times: user 192 ms, sys: 9.09 ms, total: 201 ms
Wall time: 236 ms
CPU times: user 162 ms, sys: 6.78 ms, total: 168 ms
Wall time: 182 ms


In [7]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget
import matplotlib.pyplot as plt


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.min_halo = 7.
        self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)

        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self): ##converting this to ln????
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
        
    
    def get_slope(self, Mhalo): #returns dlogMstar/dlogMhalo slope is same in log10 and ln space

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sig_lnMstar, bin_num, z): 
        lnten = np.log(10)
        logMh = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sig_lnMstar) / self.get_slope(np.asarray(logMh)) ##is this the right sigma?
        mins = logMh * lnten - plus_mins
        maxs = logMh * lnten + plus_mins
        maxMh, minMh = self.min_halo * lnten, self.max_halo * lnten
        mins[mins < maxMh] = maxMh
        maxs[maxs > minMh] = minMh
        lnMh = create_ranges_numexpr(mins, maxs, bin_num)
        dNdlnMhalo = mf.massFunction(np.e**lnMh, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        logMstar = np.apply_along_axis(self.get_Mstar, 1, lnMh/lnten)
        vals = np.zeros((bin_num,bin_num+1))
        vals[:,-1] = StellBins * lnten
        vals[:,:-1] = logMstar * lnten
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, vals, sig_lnMstar, 1)
        dNdlnMstar = np.sum(Mstar_prob * dNdlnMhalo, axis = 1) * (lnMh[:,1] - lnMh[:,0])

        return dNdlnMstar
    
    
    
    def get_dNdlnMstar(self, sig_lnMstar):
        
        if sig_lnMstar == 0.:
            self.dNdlnMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        else:
            self.dNdlnMstar = self.convolve_smhm(self.StellBins, sig_lnMstar, self.bin_num, self.z)


            
    def get_dNdlnMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdlnMbh = self.dNdlnMstar / self.m
        
    
    def get_Mdotbh(self, vals, lnxsigs, files = files):

        Mstar = vals[0]
        slope = vals[1]
        inter = vals[2]
        a = self.a
        Mbh = 10**(Mstar*slope+inter)
        
        if slope == self.slope_list[0]:
            lnxsig = lnxsigs[0]
        else:
            lnxsig = lnxsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3e38 * Mbh #ergs/s 
        #Mdotedd = Ledd / (.1 * (2.99e10)**2) #g/s
        sbhr = slope * (ssfr / 3.154e7) #1/s
        Mdotbh = sbhr * (Mbh * 2e33) #g/s
        
        
        mu_lnX = -0.5 * lnxsig**2
        mu_lnMdotbh = mu_lnX + np.log(Mdotbh) 
        
        lnMdotsig = lnxsig
        
        return mu_lnMdotbh, lnMdotsig
    
    def gauss_mdot(self, vals):
  
        x = self.lnMdotbh_list
        mu = vals[0]
        sig = vals[1]
        A = 1
        y = ( A/np.sqrt(2.0 * np.pi * sig**2.0) ) * np.exp( -(x - mu)**2.0 / (2.0 * sig**2) )

        return y
    
    def get_dNdlnL(self, lnxsigs):
        
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]
        vals = np.zeros((self.bin_num, 3))
        vals[:,0] = self.StellBins
        vals[:,1] = self.m
        vals[:,2] = b
        lnMdot_mu_msig = np.apply_along_axis(self.get_Mdotbh, 1, vals, lnxsigs)
        
        self.lnMdotbh_list = (self.LumBins + np.log10(3.9e33)) * np.log(10) - np.log(0.1*2.99e10**2)
        vals = np.zeros((self.bin_num, 2))
        vals[:,0] = lnMdot_mu_msig[:,0]
        vals[:,1] = lnMdot_mu_msig[:,1]
        
        Rl = 0.8
        Rh = 0.2
        Lc = 43.7
        Lx = np.log10(0.037*10**(self.LumBins + np.log10(3.9e33)))
        FOb = Rl * np.e**(-Lx/Lc) + Rh * (1 - np.e**(-Lx/Lc))
        
        intval = np.apply_along_axis(self.gauss_mdot, 1, vals) * (np.reshape(self.dNdlnMstar,(self.bin_num,1))) * (self.StellBins[1] - self.StellBins[0])
        self.dNdlnL = (1-FOb) * (np.sum(intval, axis = 0))
        ind_dNdlnL_off = (1-FOb) * intval
        
        self.ind_dNdlnL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdlnL_off:
            self.ind_dNdlnL[c,:] = l
            c += 1

bins = 500
fig = plt.figure(figsize=(5,5))
z=3
sig_lnX = [2.5,2.3]
dM = .2
mMid = 10.3
siglnMstar = 0.3*np.log(10)


qlf = QLF(z, bins)
qlf.get_dNdlnMstar(siglnMstar)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdlnMbh()
qlf.get_dNdlnL(sig_lnX)
    

col = np.zeros(qlf.bin_num)
col[qlf.early] = 0
col[qlf.growth] = 1
col[qlf.late] = 2


plt.plot(qlf.LumBins, np.log10(qlf.dNdlnL * np.log(10)),c='k',label='z = '+str(z), linewidth = 2.0)
plt.axis([5.8,16,-10,0])

x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='grey',label='observed')


early = np.zeros(len(qlf.ind_dNdlnL[0]))
growth = np.zeros(len(qlf.ind_dNdlnL[0]))
late = np.zeros(len(qlf.ind_dNdlnL[0]))

for l, c in zip(qlf.ind_dNdlnL, col):
    if c == 0:
        early += (l * np.log(10))
    elif c == 1:
        growth += (l * np.log(10))
    elif c == 2:
        late += (l * np.log(10))
plt.plot(qlf.LumBins, np.log10(early), linewidth = 1.0, color = 'r', label='pre-disk: early')
plt.plot(qlf.LumBins, np.log10(growth), linewidth = 1.0, color = 'b', label='post-disk: growth')
plt.plot(qlf.LumBins, np.log10(late), linewidth = 1.0, color = 'g', label='post-disk: late')


# count1 = 0
# count2 = 0
# count3 = 0
# for l, c in zip(qlf.ind_dNdlnL, col):
#     c_list = ['r','g','b']
#     if c == 0 and count1 == 0:
#         plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='pre-disk: early')
#         count1 += 1
#     elif c == 1 and count2 == 0:
#         plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='post-disk: growth')
#         count2 += 1
#     elif c == 2 and count3 == 0:
#         plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)], label='post-disk: late')
#         count3 += 1
#     else:
#         plt.plot(qlf.LumBins, np.log10(l * np.log(10)), linewidth = .75, color = c_list[int(c)])
plt.xlabel(r'$\log_{10} [L_{bol}/L_{\odot}]$')
plt.ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$')
plt.legend()
plt.tight_layout()


FigureCanvasNbAgg()

$ \frac{d\phi(L_{bol})}{d\ln{L_{bol}}} = \int p[\ln{\dot{M}_{BH}}] \frac{dN}{d\ln M_{*}} d\ln M_{*}   $

$p[\ln{\dot{M}_{BH}}](L_{bol}, M_*, z) = \frac{1}{\sqrt{2 \pi \sigma_{\ln \dot{M}_{BH}}^{2}}}exp \big(\frac{-(\ln \dot{M}_{BH} - \mu_{\ln \dot{M}_{BH}})^{2}}{2 \sigma_{\ln \dot{M}_{BH}}^{2}} \big)$

$ \sigma_{\ln{\dot{M}_{BH}}}(M_*) = \sigma_{\ln{X}}, \ \ \ \ \mu_{\ln{\dot{M}_{BH}}}(M_*,z) = \mu_{\ln{X}} + \ln{<\dot{M}_{BH}>}, \ \ \ \ \ln{\dot{M}_{BH}}(L_{bol}) = \ln{L_{bol}} - \ln{\epsilon c^2}$

$ \mu_{\ln X} = \frac{-\sigma_{\ln X}}{2} \longrightarrow \mu_{\ln \dot{M}_{BH}} = \frac{-\sigma_{\ln X}}{2} + \ln <\dot{M}_{BH}>(M_*,z) $

$  \sigma_{\ln \dot{M}_{BH}} = \sigma_{\ln X}$ :Two $\sigma_{\ln{X}}$ one pre and post disk. Which $\sigma$ used depends on $M_*$.


$    \frac{dM_{BH}}{dt} = \frac{M_{BH}}{M_{*}} \frac{d \ln M_{BH}}{d \ln M_{*}} \frac{dM_{*}}{dt}  \longrightarrow \dot{M}_{BH} = M_{BH} \frac{d\ln M_{BH}}{d\ln M_{*}} \frac{\dot{M}_*}{M_*}$

$ M_{BH}(M_*)\ , \ \ \ \ \frac{d\ln M_{BH}}{d\ln M_{*}}(M_*)\ , \ \ \ \  \frac{\dot{M}_*}{M_*}(M_*,z)\ \ \ \ \ \ \ \ \ \   \therefore \dot{M}_{BH}(M_{*},z)$

$    L_{Edd} = \epsilon \dot{M}_{Edd} c^2  = \frac{M_{BH}}{M_{\odot}} 1.26 \times 10^{38}$ erg/s $ \left( \frac{\epsilon}{1.0}\right)$

$    \eta = \frac{\dot{M}_{BH}}{\dot{M}_{Edd}} = \frac{L}{L_{Edd}} \longrightarrow L = \dot{M}_{BH}\epsilon c^2  \longrightarrow \ln L = \ln \dot{M}_{BH} + \ln [\epsilon c^2]\ \ \ \  $ where $\epsilon = 0.1$

.

pre-scaling $\epsilon$ changes:  $\ \   \dot{M}_{BH}(M_*,z)\ , \ \ \ \ c, \epsilon =$constant$,\ \ \ \ \ \ \ \ \ \  \therefore L(M_*,z) $

$ \frac{dN}{d\ln{L_Q}} = \int p[\ln{\dot{M}_{BH}}] \frac{dN}{d\ln M_{*}} d\ln M_{*}   $

$p[\ln{\dot{M}_{BH}}](L_Q, M_*, z) = \frac{1}{\sqrt{2 \pi \sigma_{\ln \dot{M}_{BH}}^{2}}}exp \big(\frac{-(\ln \dot{M}_{BH} - \mu_{\ln \dot{M}_{BH}})^{2}}{2 \sigma_{\ln \dot{M}_{BH}}^{2}} \big)$

$ \sigma_{\ln{\dot{M}_{BH}}}(M_*) = \sigma_{\ln{X}}, \ \ \ \ \mu_{\ln{\dot{M}_{BH}}}(M_*,z) = \mu_{\ln{X}} + \ln{<\dot{M}_{BH}>}, \ \ \ \ \ln{\dot{M}_{BH}}(L_Q) = \ln{L_Q} - \ln{\epsilon c^2}$

$ \mu_{\ln X} = \frac{-\sigma_{\ln X}}{2} \longrightarrow \mu_{\ln \dot{M}_{BH}} = \frac{-\sigma_{\ln X}}{2} + \ln <\dot{M}_{BH}> $

$  \sigma_{\ln \dot{M}_{BH}} = \sigma_{\ln X}$ :Two $\sigma_{\ln{X}}$ one pre and post disk. Value used depends on $M_*$.


.

.



$    <\dot{M}_{BH}> = M_{BH} \frac{dlogM_{BH}}{dlogM_{*}} <\frac{\dot{M}_*}{M_*}>    $

$    <\eta> = \frac{<\dot{M}_{BH}>}{\dot{M}_{Edd}}      $

post-scaling $\epsilon$ changes: $\dot{M}_{BH}(M_*,z)\ , \ \ \ \ c=$constant$\ ,\ \ \ \  \epsilon(\eta), \ \ \ \ \eta(\dot{M}_{BH}(M_*,z),\dot{M}_{Edd}(\epsilon, M_{BH}))$????

In [240]:
def gauss(x, var):

    mean, std, amp = var
    y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

    return y

params = [0, np.sqrt(.2), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, 1, 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, np.sqrt(5.), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)


xn = x/2
plt.plot(xn,y)
print(np.var(x), np.var(xn))


8.41708542713568 2.10427135678392


Code to create universal $<\dot{M}_{BH}>$, z evolution plot.

FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:244: RuntimeWarning: divide by zero encountered in log10
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:246: RuntimeWarning: divide by zero encountered in log10
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:251: RuntimeWarning: invalid value encountered in true_divide


Code to create universal $<\dot{M}_{BH}>$ model.

In [2]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self): ##converting this to ln????
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
        
    
    def get_slope(self, Mhalo): ##converting this to ln????

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo): ##converting this to ln????
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        
        
        Mdotbhmean = -0.5*xsig**2*np.log10(np.e)+np.log10(Mdotbh/(2*10**30))
        etamean = Mdotbhmean - np.log10(Mdotedd/(2*10**30))
        
        nsig = xsig * np.log10(eta)
        
        return etamean, nsig, Mdotbh, Mdotedd
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2], vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((1-obscured) * (np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1

bins = 500
fig = plt.figure(figsize=(5,5))
z=3
obscured = .8
sig_X = [.85,.85]
dM = 0.55
mMid = 10.3
smhm_scat = 0.3
qlf = QLF(z, bins)
qlf.get_dNdMstar(smhm_scat)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdMbh()
qlf.get_dNdL(sig_X, obscured)
col = np.zeros(qlf.bin_num)
col[qlf.early] = 0
col[qlf.growth] = 1
col[qlf.late] = 2


plt.plot(qlf.LumBins, qlf.dNdL,c='k')
plt.axis([5.8,16,-10,0])
x, y , yerr = grab_obs(z)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='blue')
for l, c in zip(qlf.ind_dNdL, col):
    c_list = ['r','gray','gray']
    plt.plot(qlf.LumBins, np.log10(l), linewidth = .5, color = c_list[int(c)])


NameError: name 'plt' is not defined

In [79]:
plt.close('all')
import matplotlib.gridspec as gridspec

def lognorm(x, sig, mean, loc):
    y = np.exp(-(np.log(x-loc)-mean)**2/(2*sig**2))/((x-loc)*sig*np.sqrt(2*np.pi))
    return y

def get_mu(mean, sig):
    return np.log(mean)-0.5*sig**2

def get_mean(mu, sig):
    return np.exp(mu+0.5*sig**2)

def get_variance(mu, sig):
    return (np.exp(sig**2)-1)*np.exp(2*mu + sig**2)


gs = gridspec.GridSpec(2, 2)
fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0,1])
ax2 = fig.add_subplot(gs[1,1])
ax3 = fig.add_subplot(gs[1,0])


s1 = .55
s2 = .85
x1 = np.linspace(-12,12,20000)
x2 = np.linspace(0,10,200000)
x3 = np.linspace(-8,0,200000)



ax3.plot(x1,qlf.gauss(x1, -0.5*s1**2*np.log10(np.e), s1*np.log10(np.e), 1),c='k')
ax3.plot(x1,qlf.gauss(x1, -0.5*s2**2*np.log10(np.e), s2*np.log10(np.e),1),c='r')
ax3.axis([-2,2,0,1.8])
ax3.set_xlabel(r'$log_{10}[X]$')
ax3.set_ylabel('Probability Density')
ax3.text(1,1.,r'$\sigma_{log_{10}X} = $'+str(s1*np.log10(np.e))[0:4],color='k')
ax3.text(1,.9,r'$\sigma_{log_{10}X} = $'+str(s2*np.log10(np.e))[0:4],color='r')
ax3.text(1,1.2,r'$\mu_{log_{10}X} = $'+str(-0.5*s1**2*np.log10(np.e))[0:5], color='k')
ax3.text(1,1.1,r'$\mu_{log_{10}X} = $'+str(-0.5*s2**2*np.log10(np.e))[0:5],color='r')
# ax3.text(-1.9,1.25,r'$\mu_{log_{10}[X]} = 0$')
# ax3.text(-1.9,1.35,r'$\sigma_{log_{10}[X]}^2 = e^{2\mu_X +\sigma_X ^2}(e^{\sigma_X ^2}-1) / ln[10]^2$')
# ax3.text(-1.9,1.5,r'$p(log_{10}[X]) = \frac{1}{\sqrt{2\pi\sigma_{log_{10}[X]}^2}}e^{-(log_{10}[X]-\mu_{log_{10}[X]})^2/(2\sigma_{log_{10}[X]}^2)}$')

print(lognorm(x2, s1, -0.5*s1**2, 0))

ax.plot(x2, lognorm(x2, s1, -0.5*s1**2, 0), c = 'k', label=r'pre-disk')
ax.plot(x2, lognorm(x2, s2, -0.5*s2**2, 0), c= 'r', label=r'post-disk')
ax.axis([0,6,0,1.5])
ax.set_xlabel(r'$X$')
ax.set_ylabel('Probability Density')
ax.text(2,1.,r'$\sigma_{lnX} = $'+str(s1),color='k')
ax.text(2,.9,r'$\sigma_{lnX} = $'+str(s2),color='r')
ax.text(2,1.1,r'$\mu_{lnX} = - \sigma_{lnX}/2$')
# ax.text(2,1.1,r'$\mu_{lnX} = ?$')
# ax.text(-2.8,1,r'$p(X) = \frac{e^{-(ln(X)-\mu_{lnX})^2/(2\sigma_{lnX} ^2)}}{X\sigma_{lnX}\sqrt{2 \pi}}$',fontsize=13)
# ax.text(-2.7,.9,r'$<X> \equiv 1$')
# ax.text(-2.8,.8,r'$<X> = \int_{0}^{\infty}$ X p(X) dX')
# ax.text(-1.8,.7,r'$= e^{\mu_{lnX} + \sigma_{lnX}/2}$')
# ax.text(-2.4,.6,r'$\mu_{lnX} = ln[1] - \sigma_{lnX}/2$')
# ax.text(-1.8,.5, r'$ = - \sigma_{lnX}/2$')
ax.legend(loc='upper center', bbox_to_anchor=(0.75, .98))
ax.axvline(1, color = 'gray', linestyle='dotted')

plt.suptitle(r'$X = \frac{\dot{M}_{BH}}{<\dot{M}_{BH}>}$     Universal Relation')



ax1.plot(x1,qlf.gauss(x1, -0.5*s1**2, s1, 1),c='k')
ax1.plot(x1,qlf.gauss(x1, -0.5*s2**2, s2,1),c='r')
ax1.axis([-2,2,0,1.8])
ax1.set_xlabel(r'$ln[X]$')
ax1.set_ylabel('Probability Density')
ax1.text(1,1.,r'$\sigma_{lnX} = $'+str(s1)[0:4],color='k')
ax1.text(1,.9,r'$\sigma_{lnX} = $'+str(s2)[0:4],color='r')
ax1.text(1,1.2,r'$\mu_{lnX} = $'+str(-0.5*s1**2)[0:5], color='k')
ax1.text(1,1.1,r'$\mu_{lnX} = $'+str(-0.5*s2**2)[0:5],color='r')

for mdot, c in zip([-1,-2,-3,-4,-5],['r','orange','gold','g','b']):
    ax2.plot(x1, qlf.gauss(x1, -0.5*s1**2*np.log10(np.e)+mdot, s1*np.log10(np.e), 1),c=c,linestyle='dashed',label = r'pre-$log_{10}[<\dot{M}_{BH}>] = $'+str(mdot))
    ax2.plot(x1, qlf.gauss(x1, -0.5*s2**2*np.log10(np.e)+mdot, s2*np.log10(np.e), 1),c=c,linestyle='solid',label = r'post-$log_{10}[<\dot{M}_{BH}>] = $'+str(mdot))

ax2.axis([-10,0,0,1.8])
ax2.set_xlabel(r'$log_{10}[\dot{M}_{BH}] \ \ yr^{-1}$')
ax2.set_ylabel('Probability Density')
ax2.legend()

fig.savefig('plots/universaldist_mdot_v2.3.pdf')

FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: divide by zero encountered in log
  """
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: divide by zero encountered in true_divide
  """
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:5: RuntimeWarning: invalid value encountered in true_divide
  """


[           nan 7.77059197e-65 8.89971498e-56 ... 3.45334061e-06
 3.45318328e-06 3.45302595e-06]


In [16]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
            
        closest_a = np.argmin(np.abs(a_list - a))
        masses = np.array(mass_list[closest_a])
        ssfrs = np.array(ssfr_list[closest_a])
        closest_m = np.argmin(np.abs(masses - Mstar))
        nonzero = (ssfrs != 0)
        minm = np.min(masses[nonzero])
        maxm = np.max(masses[nonzero])
        if minm < Mstar < maxm:
            ssfr = np.interp(Mstar, masses[nonzero], ssfrs[nonzero])
        else:
            ssfr = ssfr_list[closest_a][closest_m]
        
        flag = 0
        if Mstar > maxm:
            flag = 1


        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig
        
        return np.log10(eta), nsig, Mdotbh, Mdotedd, Mstar, ssfr, flag, Mbh, sbhr
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1


In [41]:
plt.close('all')
bins = 50
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(8,4),sharey=True)
obscured = .8
sig_X = [1.25,.85]
dM = .55
mMid = 10.3
smhm_scat = 0.3
for z, c in zip([0.5,1.0,1.5,2.0,2.5,3.0,4.0],['r','orange','gold','green','blue','violet','indigo']):
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(smhm_scat)
    qlf.get_SMBM(dM, mMid)
    qlf.get_dNdMbh()
    qlf.get_dNdL(sig_X, obscured)

    logMstar = qlf.plotpurp[:,4]
    ssfr = qlf.plotpurp[:,5] #yr^-1
    sfr = ssfr * 10**logMstar
    Mdotbh = qlf.plotpurp[:,2]/(2e30*3.17098e-8)
    flags = qlf.plotpurp[:,6]
    logMbh = qlf.plotpurp[:,7]
    sbhr = qlf.plotpurp[:,8]
    lw = 1. 
    
    ax1.plot(logMstar, np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax1.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax1.set_ylabel(r'$log_{10}[<\dot{M}_{BH}>/M_{\odot}] \ \ yr^{-1}$')

    
    ax2.plot(np.log10(sfr), np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax2.set_xlabel(r'$log_{10}[<\dot{M}_{*}>/M_{\odot}]$')

plt.tight_layout()
ax1.legend()
fig.savefig('plots/MdotBHvsMstarSFR.pdf')

FigureCanvasNbAgg()

In [30]:
 import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib
plt.close('all')
bins = 200
fig, ax = plt.subplots(figsize=(6,6))
cbax = fig.add_axes([0.05, 0.35, .9, 0.05]) 
fig.subplots_adjust(bottom=0.5)
dM = .5
obscured = .75
zlist = np.linspace(0,6,800)
mulist = []
cut = []
for z in zlist:
    obscured = .75
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mu = np.log10(qlf.plotpurp[:,2]/2e30*3.154e7)
    mulist.append(mu)
    cut.append(np.argmin(qlf.early))

cutlist = []
for i, mul in zip(cut,mulist):
    cutlist.append(mul[i])
    
norm = matplotlib.colors.Normalize(vmin=min(qlf.StellBins), vmax=max(qlf.StellBins))
cb = matplotlib.colorbar.ColorbarBase(cbax, cmap='inferno', norm=norm, orientation='horizontal')
cb.set_label(r'$log_{10}[M_{*}]$')

i_mulist = np.transpose(mulist)
for i, j in zip(i_mulist,qlf.StellBins):
    ax.plot(zlist, i, c = cm.inferno(norm(j)),linewidth = 0.5)

ax.set_ylabel(r'$log_{10} [<\dot{M}_{BH}>/M_{\odot}] yr^{-1}$')
ax.set_xlabel(r'z')
ax.plot(zlist,cutlist,c='k',label='critical mass',linestyle='dotted')
ax.legend()
fig.suptitle(r'Redshift evolution of <$\dot{M}_{BH}$> for varrying $M_{*}$')

fig.show()
#fig.savefig('plots/Mdot_z_evolution.pdf')


FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:246: RuntimeWarning: divide by zero encountered in log10
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:21: RuntimeWarning: divide by zero encountered in log10


In [21]:
vals = np.loadtxt("Quest-emcee/output/dM-chain-v1.0_n200_s1000_t12.dat")
print(max(vals[0]))

corner.corner(vals.reshape((-1,1)), labels = ['dM'])
max(vals.reshape((-1,1)))

0.9218208089348593


FigureCanvasNbAgg()

array([0.92182081])

In [4]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
from scipy.optimize import newton


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.txt", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.min_halo = 7.
        self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(self.min_halo, self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)

        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.StellBins = np.linspace(8.0, self.max_stell, bin_num)
        
        
    def get_zparams(self): ##converting this to ln????
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
        
        
    
    def get_slope(self, Mhalo): #returns dlogMstar/dlogMhalo slope is same in log10 and ln space

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid = 10.3, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2]
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        
        self.m = np.zeros(self.bin_num)
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        
        self.b = np.zeros(self.bin_num)
        self.b[self.early] = self.int_list[0]
        self.b[self.growth] = self.int_list[1]
        self.b[self.late] = self.int_list[2]
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sig_lnMstar, bin_num, z):
        lnten = np.log(10)
        logMh = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sig_lnMstar) / self.get_slope(np.asarray(logMh)) ##is this the right sigma?
        mins = logMh * lnten - plus_mins
        maxs = logMh * lnten + plus_mins
        maxMh, minMh = self.min_halo * lnten, self.max_halo * lnten
        mins[mins < maxMh] = maxMh
        maxs[maxs > minMh] = minMh
        lnMh = create_ranges_numexpr(mins, maxs, bin_num)
        dNdlnMhalo = mf.massFunction(np.e**lnMh, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        logMstar = np.apply_along_axis(self.get_Mstar, 1, lnMh/lnten)
        vals = np.zeros((bin_num,bin_num+1))
        vals[:,-1] = StellBins * lnten
        vals[:,:-1] = logMstar * lnten
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, vals, sig_lnMstar, 1)
        dNdlnMstar = np.sum(Mstar_prob * dNdlnMhalo, axis = 1) * (lnMh[:,1] - lnMh[:,0])

        return dNdlnMstar
    
    
    
    def get_dNdlnMstar(self, sig_lnMstar):
        
        if sig_lnMstar == 0.:
            self.dNdlnMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')
        else:
            self.dNdlnMstar = self.convolve_smhm(self.StellBins, sig_lnMstar, self.bin_num, self.z)


        
    
    def get_Mdotbh(self, vals, files = files):

        Mstar = vals[0]
        slope = vals[1]
        inter = vals[2]
        lnxsig = vals[3]
        a = self.a
        Mbh = 10**(Mstar*slope+inter)
        
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3e38 * Mbh #ergs/s 
        Mdotedd = Ledd / (.1 * (2.99e10)**2) #g/s
        sbhr = slope * (ssfr / 3.154e7) #1/s
        Mdotbh = sbhr * (Mbh * 2e33) #g/s
        
        
        mu_lnX = -0.5 * lnxsig**2
        mu_lnMdotbh = mu_lnX + np.log(Mdotbh) 
        
        lnMdotsig = lnxsig
        
        return mu_lnMdotbh, lnMdotsig, np.log(Mdotedd)
    
    
    def gauss_Mdot(self, lnMdotbh):
  
        x = lnMdotbh
        mu = self.Mdot_mu_sig[:,0]
        sig = self.Mdot_mu_sig[:,1]
        A = 1
        y = ( A/np.sqrt(2.0 * np.pi * sig**2.0) ) * np.exp( -(x - mu)**2.0 / (2.0 * sig**2) )

        return y
    
    
    def get_dNdlnL(self, lnxsigs):
        
        lnxsig_list = np.zeros(self.bin_num)
        lnxsig_list[self.early] = lnxsigs[0]
        lnxsig_list[self.growth] = lnxsigs[1]
        lnxsig_list[self.late] = lnxsigs[1]
        tenper = int(self.bin_num * 0.1)
        tranpoint = np.argmin(self.early)
        lintrans = np.linspace(lnxsigs[0], lnxsigs[1], tenper * 2, endpoint = False)
        lnxsig_list[tranpoint - tenper : tranpoint + tenper] = lintrans
        
        vals = np.zeros((self.bin_num, 4))
        vals[:,0] = self.StellBins
        vals[:,1] = self.m
        vals[:,2] = self.b
        vals[:,3] = lnxsig_list
        self.Mdot_mu_sig = np.apply_along_axis(self.get_Mdotbh, 1, vals)
        
        self.lnMdotbh_list = (self.LumBins + np.log10(3.9e33)) * np.log(10) - np.log(0.1*2.99e10**2)
        
        Rl = 0.8
        Rh = 0.2
        Lc = 10**43.7
        Lx = 0.037*10**(self.LumBins + np.log10(3.9e33))
        self.FOb = Rl * np.e**(-Lx/Lc) + Rh * (1 - np.e**(-Lx/Lc))
        
        self.dNdlnL = (1-self.FOb) * np.sum(np.apply_along_axis(self.gauss_Mdot, 1, self.lnMdotbh_list.reshape(len(self.lnMdotbh_list),1)) * self.dNdlnMstar * (self.StellBins[1] - self.StellBins[0]), axis = 1)
    
        

In [5]:
import matplotlib.pyplot as plt
%matplotlib widget
fig = plt.figure(figsize=(10,10))

x, y , yerr = grab_obs(1.5)
plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5, c='r', label = 'observed')
plt.legend()

def go():
    qlf = QLF(1.5, 1000)
    qlf.get_dNdlnMstar(0.7)
    qlf.get_SMBM(.3)
    qlf.LumBins = np.linspace(6,16,1000)
    qlf.get_dNdlnL([6,2.8])
    return qlf.LumBins, np.log10(qlf.dNdlnL * np.log(10))

qlf = QLF(1.5, 1000)
qlf.get_dNdlnMstar(0.7)
qlf.get_SMBM(.3)
qlf.LumBins = np.linspace(6,16,1000)
#%timeit qlf.get_dNdlnL([6,2.8])

qlf = QLF(1.5, 1000)
qlf.get_dNdlnMstar(0.7)
qlf.get_SMBM(.3)
qlf.LumBins = np.array(sorted(x))
#%timeit qlf.get_dNdlnL([6,2.8])

def go1():
    qlf = QLF(1.5, 1000)
    qlf.get_dNdlnMstar(0.7)
    qlf.get_SMBM(0.3)
    qlf.LumBins = np.array(sorted(x))
    qlf.get_dNdlnL([6,2.8])
    return qlf.LumBins, np.log10(qlf.dNdlnL * np.log(10))

xm,ym = go()
plt.plot(xm, ym, c='k')
xm,ym = go1()
plt.scatter(xm, ym, s=25, c='c')
plt

FigureCanvasNbAgg()

<module 'matplotlib.pyplot' from '/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/matplotlib/pyplot.py'>

In [11]:
import matplotlib.pyplot as plt
fig2 = plt.figure()
for star in np.linspace(8, 11.7, 8):
    Mhalo = qlf.get_Mhalo(np.asarray(star))
    sig = 3
    pm = (5.0 * sig) / qlf.get_slope(np.asarray(Mhalo))
    if qlf.get_Mstar(np.asarray(Mhalo+pm)) >= 11.7 and qlf.get_Mstar(np.asarray(Mhalo-pm)) >= 7.0:
        halos = np.linspace(Mhalo - pm, qlf.get_Mhalo(np.asarray(11.7)), 1000)
    elif qlf.get_Mstar(np.asarray(Mhalo-pm)) <= 7.0 and qlf.get_Mstar(np.asarray(Mhalo+pm)) <= 11.7:
        halos = np.linspace(qlf.get_Mhalo(np.asarray(7.0)), Mhalo + pm , 1000)
    elif qlf.get_Mstar(np.asarray(Mhalo-pm)) <= 7.0 and qlf.get_Mstar(np.asarray(Mhalo+pm)) >= 11.7:
        halos = np.linspace(qlf.get_Mhalo(np.asarray(7.0)), qlf.get_Mhalo(np.asarray(11.7)), 1000)
    else:
        halos = np.linspace(Mhalo - pm, Mhalo+pm, 1000)
    
    stars = qlf.get_Mstar(halos)
    y = 1/np.sqrt(2.0 * np.pi * sig**2.0) * np.exp( -(stars - star)**2.0 / (2.0 * sig**2))
    plt.plot(stars, y)
    plt.xlim([0,20])

FigureCanvasNbAgg()